# 🔍 Phase 6: SQL Analytics & Business Queries
**Project:** AI-Powered Customer Intelligence & Revenue Optimization Platform  
**Analyst:** Naman Roy  
**Database:** SQLite (`olist.db`)

---
### Objective
Answer key business and executive questions using standard SQL queries, Window Functions, CTEs, and aggregations joining on persistent unique customers.


In [ ]:
import sqlite3
import pandas as pd
import os

db_path = '../../02_Data/olist.db'
conn = sqlite3.connect(db_path)
print("✅ Connected to SQLite database:", db_path)


## Query 1: Executive KPI Overview

In [ ]:
q1 = """
SELECT 
    COUNT(DISTINCT o.order_id)           AS total_orders,
    COUNT(DISTINCT c.customer_unique_id) AS total_unique_customers,
    ROUND(SUM(p.payment_value), 2)       AS total_revenue,
    ROUND(AVG(p.payment_value), 2)       AS avg_payment_transaction
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered';
"""
pd.read_sql(q1, conn)


## Query 2: Monthly Revenue & Order Velocity

In [ ]:
q2 = """
SELECT 
    strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
    COUNT(DISTINCT o.order_id)                     AS total_orders,
    ROUND(SUM(p.payment_value), 2)                 AS monthly_revenue
FROM orders o
JOIN payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY order_month
ORDER BY order_month;
"""
pd.read_sql(q2, conn).tail(10)


## Query 3: Top Revenue States & SP Dominance

In [ ]:
q3 = """
SELECT 
    c.customer_state               AS state,
    COUNT(DISTINCT o.order_id)     AS total_orders,
    ROUND(SUM(p.payment_value), 2) AS total_revenue,
    ROUND(SUM(p.payment_value) * 100.0 / 
        (SELECT SUM(payment_value) FROM payments p2 
         JOIN orders o2 ON p2.order_id = o2.order_id 
         WHERE o2.order_status = 'delivered'), 2) AS revenue_share_pct
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY state
ORDER BY total_revenue DESC
LIMIT 10;
"""
pd.read_sql(q3, conn)


## Query 4: Top Customers by Lifetime Spend (Grouped by `customer_unique_id`)

In [ ]:
q4 = """
SELECT 
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id)     AS lifetime_orders,
    ROUND(SUM(p.payment_value), 2) AS lifetime_spend
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
ORDER BY lifetime_spend DESC
LIMIT 10;
"""
pd.read_sql(q4, conn)


## Query 5: Category Review Ratings using CTE & Window Functions

In [ ]:
q5 = """
WITH category_perf AS (
    SELECT 
        p.product_category_name        AS category,
        COUNT(DISTINCT o.order_id)     AS order_count,
        ROUND(AVG(r.review_score), 2)  AS avg_review_score
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p     ON oi.product_id = p.product_id
    JOIN reviews r      ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered' AND p.product_category_name IS NOT NULL
    GROUP BY category
)
SELECT 
    category,
    order_count,
    avg_review_score,
    DENSE_RANK() OVER (ORDER BY avg_review_score DESC) AS quality_rank
FROM category_perf
WHERE order_count >= 200
ORDER BY avg_review_score DESC
LIMIT 10;
"""
pd.read_sql(q5, conn)


In [ ]:
conn.close()
print("✅ SQL queries completed successfully.")
